<a href="https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/eda_esda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multidimensional poverty in Bolivia: exploratory and spatial data analysis

Part of the [GeoDevelopment Observatory of Bolivia](https://quarcs-lab.github.io/gdo4bolivia/).

This notebook combines classical exploratory data analysis (EDA) with exploratory spatial data analysis (ESDA) for Bolivia's 343 municipalities and indigenous autonomies. The outcome is the **Multidimensional Poverty Index** (`sdg1_2_mpi`) of the Atlas de los ODS en Bolivia 2026, which runs from 0 to 1: **higher values mean more poverty**. The analysis centres on 2012, when night-time lights begin, and repeats the main steps for 2024. Night-time lights per capita, census population and a set of infrastructure and connectivity indicators are the companions.

1. **Data**: everything is loaded with the [`geo4bolivia`](https://quarcs-lab.github.io/geo4bolivia/) package
2. **EDA**: summaries, distributions, department comparisons and change since 2012, population, night-time lights and infrastructure correlates
3. **Maps**: both periods in common classes, and night-time lights
4. **Spatial dependence**: spatial weights, global and local Moran's I, how the clusters changed, bivariate Moran's I and bivariate LISA
5. **Spatial inequality**: Gini, Theil and its decomposition by department, spatial Gini
6. **Spatial heterogeneity**: a multiscale geographically weighted regression (MGWR) of poverty on night-time lights
7. **Summary**

Almost every step is one function of the package, which returns a result with its figure, its table and a reading of the numbers. The exceptions are the bivariate Moran statistics of section 4.4, which the package does not have: they use the esda library, with weights and boundaries from the package. The notebook began as a port of DS4Bolivia's *Extended EDA + Spatial Analysis* notebook. The focused notebooks `esda_dependence`, `esda_inequality` and `esda_heterogeneity` go deeper into sections 4–6.

## 0. Setup

In Google Colab the next cell installs the package with its MGWR extra (it brings the mgwr library; Colab already has the rest). Locally, `uv sync` has installed everything already.

In [1]:
import importlib.util

if any(importlib.util.find_spec(p) is None for p in ["geo4bolivia", "mgwr"]):
    !pip install -q "geo4bolivia[mgwr]"

In [2]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from esda.moran import Moran_BV, Moran_Local_BV          # section 4.4 only
from libpysal.weights import lag_spatial

import geo4bolivia as geo

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"     # also in the worker processes that MGWR starts
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
# Plotly: Colab's own renderer there; elsewhere load plotly.js from its CDN to keep the file small
pio.renderers.default = "colab" if "google.colab" in sys.modules else "notebook_connected"
print("geo4bolivia", geo.__version__)

geo4bolivia 0.8.0


## 1. Data

Everything is at the municipality scale and keyed by `geo_id`:

| Variable | Dataset | Years | Meaning |
| --- | --- | --- | --- |
| `sdg1_2_mpi` | `sdg01` | 2012, 2024 | Multidimensional Poverty Index (0–1, higher = poorer) |
| `population` | `sdg` | 2012, 2024 | Census population |
| `ln_t400NTLpc` | `ntl` | 2012, 2020 | Hodrick–Prescott trend of the log of night-time lights per capita |
| infrastructure and connectivity | `sdg07`, `sdg08`, `sdg09`, `sdg11`, `sdg` | 2012, 2024 | see section 2.5 |

The Atlas de los ODS en Bolivia 2026 covers all **343 municipalities and indigenous autonomies** at two points in time, near the 2012 and the 2024 censuses (stored as years 2012 and 2024). Night-time lights per capita are built from gridded population on the older municipal map of 2015, `map339`, and cover its **339 units**: the four units created since (three indigenous autonomies, TIOC, and San Pedro de Macha) have no lights value, and the six municipalities that ceded territory to them are described by their 2015 territory. The two maps share geo IDs, so the datasets join on `geo_id`. Analyses that involve lights use those 339 units on the 2015 map (`map="map339"`); the rest use all 343 on the 2025 map.

The Atlas is published as one dataset per goal (`sdg01` to `sdg17`) plus `sdg` (the overall index, census population and urbanization). `geo.combine` puts the variables of several datasets in one panel, one row per unit and period. Two choices are made here:

- The lights of 2012 go with the poverty of 2012, and the lights of 2020 (the last year of the series) with the poverty of 2024: the lights of 2020 are stored in the rows of 2024.
- The Atlas gives the urbanization rate for 2024 only; it is used in both periods.

In [3]:
infra = ["urbanization", "sdg7_1_cae", "sdg9_8_tfc", "sdg9_8_ci", "sdg9_8_drb", "sdg11_2_atc",
         "sdg8_10_dsb", "sdg9_1_vpc"]
df = geo.combine({"sdg01": ["sdg1_2_mpi"], "sdg": ["population", "urbanization"],
                  "sdg07": ["sdg7_1_cae"], "sdg08": ["sdg8_10_dsb"], "sdg11": ["sdg11_2_atc"],
                  "sdg09": ["sdg9_8_tfc", "sdg9_8_ci", "sdg9_8_drb", "sdg9_1_vpc"]},
                 scale="map343")
df["urbanization"] = df.groupby("geo_id")["urbanization"].transform("first")   # the 2024 value

ntl = geo.load("ntl", variables="ln_t400NTLpc", years=[2012, 2020])
ntl["year"] = ntl["year"].replace({2020: 2024})      # lights of 2020 next to poverty of ~2024
df = geo.add_names(df.merge(ntl, on=["geo_id", "year"], how="left"), groups=["department"])
df["ln_pop"] = np.log(df["population"])
labels = {**geo.labels(df), "ln_pop": "Log of population"}

lit = df.dropna(subset=["ln_t400NTLpc"])             # the 339 units with lights (map339)
dark = df.loc[df["ln_t400NTLpc"].isna(), "name"].unique()
print(df["geo_id"].nunique(), "units,", lit["geo_id"].nunique(), "with lights | without:",
      ", ".join(dark))
df[["geo_id", "name", "department", "year", "sdg1_2_mpi", "population", "ln_t400NTLpc"]].head()

343 units, 339 with lights | without: TIOC-Raqaypampa, TIOC-Territorio Indígena Multiétnico, TIOC-Jatun Ayllu Yura, San Pedro de Macha


,geo_id,name,department,year,sdg1_2_mpi,population,ln_t400NTLpc
0,BO-01-01-01,Azurduy,Chuquisaca,2012,0.512,10652,11.048312
1,BO-01-01-01,Azurduy,Chuquisaca,2024,0.187,7903,13.136794
2,BO-01-01-02,Tarvita,Chuquisaca,2012,0.452,14261,12.329279
3,BO-01-01-02,Tarvita,Chuquisaca,2024,0.133,16812,12.656998
4,BO-01-02-01,Villa Serrano,Chuquisaca,2012,0.267,11161,12.815315


## 2. Exploratory data analysis

### 2.1 Descriptive statistics

What do we know about the centre and spread of poverty, and how has it changed? `geo.describe` summarises each variable in each period, and `geo.extremes` names the units at both ends.

In [4]:
geo.describe(df, ["sdg1_2_mpi", "ln_t400NTLpc", "population"])

,variable,label,unit,year,count,mean,std,min,p25,median,p75,max
0,sdg1_2_mpi,Multidimensional poverty index (0-1),index (0-1),2012,343,0.234259,0.105528,0.018000,0.153000,0.227000,0.306500,5.630000e-01
1,sdg1_2_mpi,Multidimensional poverty index (0-1),index (0-1),2024,343,0.063262,0.044054,0.003000,0.030000,0.053000,0.083000,2.960000e-01
2,ln_t400NTLpc,Log night-time lights per capita (trend),log(10⁸ × nW/cm²/sr per person),2012,339,13.033028,1.671621,0.856095,12.050438,13.118243,14.268402,1.707105e+01
3,ln_t400NTLpc,Log night-time lights per capita (trend),log(10⁸ × nW/cm²/sr per person),2024,339,14.039931,1.177322,10.488098,13.205945,14.101274,14.922364,1.806340e+01
4,population,Population (census),people,2012,343,29329.026239,107270.841863,509.000000,5945.000000,10647.000000,19544.000000,1.444425e+06
5,population,Population (census),people,2024,343,33133.915452,116499.083621,875.000000,6691.500000,12296.000000,21083.000000,1.610982e+06


In [5]:
geo.extremes(df, "sdg1_2_mpi", year=2012, n=10)

**Multidimensional poverty index (0-1)** in 2012 ranges from 0.018 (Nuestra Señora de La Paz) to 0.563 (TIOC-Raqaypampa) across the 343 units with data; the median unit has 0.227.

The 10 highest: TIOC-Raqaypampa (0.563), Poroma (0.541), Azurduy (0.512), Sena (0.507), San Pedro (0.496), Tacobamba (0.483), Alalay (0.479), Ckochas (0.475), San Pedro (0.472), Incahuasi (0.462).

The 10 lowest: Nuestra Señora de La Paz (0.018), Huachacalla (0.026), Colcapirhua (0.039), Tarija (0.045), Oruro (0.046), Santa Cruz de la Sierra (0.051), Potosí (0.051), Bermejo (0.053), El Alto (0.053), Punata (0.059).

The gap between the two ends is 0.545. Extreme values deserve a second look before any analysis: they may be the most interesting units, very small units whose rates are noisy, or errors in the data.

_These are associations, not causal effects. A causal reading needs a research design._

In [6]:
ends24 = geo.extremes(df, "sdg1_2_mpi", year=2024, n=5).df
print(ends24[["name", "department", "sdg1_2_mpi", "end"]].to_string(index=False))

chg = geo.change(df, "sdg1_2_mpi")
print(f"\nMPI fell in {(chg['sdg1_2_mpi_change'] < 0).sum()} of {len(chg)} units; "
      f"mean change {chg['sdg1_2_mpi_pct_change'].mean():.1f}%")
quarter = pd.qcut(chg["sdg1_2_mpi_2012"], 4, labels=["least poor", "second", "third", "poorest"])
print("Mean change (%) by quarter of the 2012 values:",
      chg.groupby(quarter, observed=True)["sdg1_2_mpi_pct_change"].mean().round(1).to_dict())
print(f"Spearman correlation of the two periods: "
      f"{geo.rank_mobility(df, 'sdg1_2_mpi').spearman:.3f}")
dark12 = geo.extremes(df, "ln_t400NTLpc", year=2012, n=2).df
print("Darkest units in 2012:",
      dark12.loc[dark12["end"] == "lowest", ["name", "ln_t400NTLpc"]].round(2).values.tolist())

                    name department  sdg1_2_mpi     end
         TIOC-Raqaypampa Cochabamba       0.296 highest
                  Poroma Chuquisaca       0.223 highest
               San Pedro     Potosí       0.213 highest
                El Choro      Oruro       0.203 highest
               Vila Vila Cochabamba       0.200 highest
                 El Alto     La Paz       0.006  lowest
 Santa Cruz de la Sierra Santa Cruz       0.006  lowest
                  Trigal Santa Cruz       0.006  lowest
             Colcapirhua Cochabamba       0.005  lowest
Nuestra Señora de La Paz     La Paz       0.003  lowest

MPI fell in 343 of 343 units; mean change -74.6%
Mean change (%) by quarter of the 2012 values: {'least poor': -78.8, 'second': -75.0, 'third': -74.8, 'poorest': -69.7}


Spearman correlation of the two periods: 0.861


Darkest units in 2012: [['San Lorenzo', 5.47], ['Mojinete', 0.86]]


In 2012 the average municipality has an MPI of 0.234 (median 0.227, standard deviation 0.106), with values from 0.018 in Nuestra Señora de La Paz to 0.563 in TIOC-Raqaypampa (Cochabamba), one of the new indigenous autonomies. The poorest ten are rural municipalities of Chuquisaca (Poroma, Azurduy, Incahuasi), Potosí (San Pedro, Tacobamba, Ckochas), Cochabamba (Raqaypampa, Alalay) and Pando (Sena, San Pedro). The least poor are the large cities (La Paz, El Alto, Santa Cruz de la Sierra, Oruro, Tarija, Potosí), Colcapirhua and Punata in the Cochabamba valley, Bermejo, and Huachacalla (Oruro).

By 2024 poverty has fallen in every one of the 343 units, by about three quarters on average: the average MPI is 0.063 (median 0.053, standard deviation 0.044), from 0.003 in La Paz to 0.296, again in Raqaypampa. The ranking is persistent (Spearman correlation 0.861 between the two periods): Raqaypampa, Poroma and San Pedro (Potosí) are still the three poorest, joined by El Choro (Oruro) and Vila Vila (Cochabamba).

Poverty fell proportionally less where it was highest: by 69.7% on average in the poorest quarter of 2012, against 78.8% in the least poor quarter.

Night-time lights have one extreme value: Mojinete (Potosí) has a log of 0.86 in 2012, against 5.47 for the next darkest municipality.

How do departments compare? The table gives the simple mean over municipalities and, with `geo.aggregate`, the mean weighted by census population (the poverty of the average resident), in both periods. The weights are the rule the metadata declares for the variable.

In [7]:
simple = df.pivot_table(index="department", columns="year", values="sdg1_2_mpi", aggfunc="mean")
weighted = geo.add_names(geo.aggregate(df[["geo_id", "year", "sdg1_2_mpi"]], to="department",
                                       dataset="sdg01"))
weighted = weighted.pivot(index="name", columns="year", values="sdg1_2_mpi")

by_dep = pd.DataFrame({"units": df[df["year"] == 2012].groupby("department").size(),
                       "mean 2012": simple[2012], "weighted 2012": weighted[2012],
                       "mean 2024": simple[2024], "weighted 2024": weighted[2024]})
by_dep["change in weighted (%)"] = 100 * (by_dep["weighted 2024"] / by_dep["weighted 2012"] - 1)
by_dep.sort_values("mean 2012", ascending=False).round(3)

,units,mean 2012,weighted 2012,mean 2024,weighted 2024,change in weighted (%)
Chuquisaca,29,0.323,0.221,0.090,0.058,-73.625
Pando,15,0.315,0.236,0.068,0.049,-79.167
Potosí,42,0.279,0.224,0.083,0.063,-71.933
Beni,20,0.250,0.229,0.075,0.072,-68.295
Oruro,35,0.235,0.129,0.069,0.037,-71.373
Cochabamba,48,0.230,0.142,0.070,0.038,-73.240
La Paz,87,0.224,0.110,0.060,0.027,-74.948
Santa Cruz,56,0.169,0.095,0.032,0.017,-81.933
Tarija,11,0.118,0.079,0.027,0.018,-77.728


In 2012 Chuquisaca (simple mean 0.323), Pando (0.315) and Potosí (0.279) are the poorest departments, and Tarija (0.118) and Santa Cruz (0.169) the least poor. Weighting by population lowers every department's figure, often by about half (La Paz from 0.224 to 0.110, Oruro from 0.235 to 0.129), because the populous municipalities are the least poor. It also reorders the top: weighted, Pando (0.236), Beni (0.229), Potosí (0.224) and Chuquisaca (0.221) are almost level, since Beni's figure barely moves.

Between the periods the population-weighted MPI falls by 68% (Beni) to 82% (Santa Cruz). In 2024 Beni (0.072), Potosí (0.063) and Chuquisaca (0.058) are the poorest departments for the average resident, and Santa Cruz (0.017) and Tarija (0.018) the least poor.

### 2.2 Distributions and department differences

How is poverty distributed across the 343 municipalities and the 9 departments? `geo.distribution` overlays the two periods, and with `by="department"` it compares the departments.

In [8]:
geo.distribution(df, "sdg1_2_mpi")

Between 2012 and 2024, the mean of **Multidimensional poverty index (0-1)** fell from 0.234 to 0.0633, and the median went from 0.227 to 0.053.

The spread across units narrowed: the standard deviation went from 0.106 to 0.0441, and the interquartile range (the middle half of the units) from 0.153 to 0.053.

**Multidimensional poverty index (0-1)** in 2024 averages 0.0633 across 343 units (median 0.053, standard deviation 0.0441). Half of the units lie between 0.03 and 0.083, and the full range runs from 0.003 to 0.296. The distribution is strongly right-skewed: a tail of high values pulls the mean above the median (skewness 1.44).

_These are associations, not causal effects. A causal reading needs a research design._

In [9]:
box12 = geo.distribution(df, "sdg1_2_mpi", year=2012, by="department", kind="box")
box12

By Department in 2012 (9 groups), the median of **Multidimensional poverty index (0-1)** is highest in **Pando** (0.336) and lowest in **Tarija** (0.139).

Units differ most within **Cochabamba** (standard deviation 0.126) and least within **Tarija** (0.0516). Where the groups' distributions barely overlap, the differences between groups are large compared with those within them.

In each box, the line is the median and the box holds the middle half of the units; points beyond the whiskers are unusually high or low units.

_These are associations, not causal effects. A causal reading needs a research design._

In [10]:
box24 = geo.distribution(df, "sdg1_2_mpi", year=2024, by="department", kind="box")
columns = ["department", "min", "median", "max"]
box12.df[columns].merge(box24.df[columns], on="department", suffixes=(" 2012", " 2024")).round(3)

,department,min 2012,median 2012,max 2012,min 2024,median 2024,max 2024
0,Beni,0.145,0.227,0.460,0.035,0.062,0.168
1,Chuquisaca,0.071,0.317,0.541,0.014,0.074,0.223
2,Cochabamba,0.039,0.222,0.563,0.005,0.056,0.296
3,La Paz,0.018,0.222,0.434,0.003,0.056,0.144
4,Oruro,0.026,0.232,0.443,0.007,0.066,0.203
5,Pando,0.112,0.336,0.507,0.019,0.056,0.177
6,Potosí,0.051,0.270,0.496,0.012,0.078,0.213
7,Santa Cruz,0.051,0.154,0.338,0.006,0.027,0.076
8,Tarija,0.045,0.139,0.201,0.010,0.023,0.045


In [11]:
geo.distribution(df, "ln_t400NTLpc", year=2012)

**Log night-time lights per capita (trend)** in 2012 averages 13 across 339 units (median 13.1, standard deviation 1.67). Half of the units lie between 12.1 and 14.3, and the full range runs from 0.856 to 17.1. The distribution is strongly left-skewed: a tail of low values pulls the mean below the median (skewness -1.66).

_These are associations, not causal effects. A causal reading needs a research design._

In 2012 poverty is mildly right-skewed (mean 0.234 above the median 0.227, skewness 0.46). By 2024 the whole distribution has moved towards zero, narrowed and become more skewed (mean 0.063, median 0.053, skewness 1.44), with a thin upper tail. Log night-time lights are left-skewed (−1.65), with Mojinete and a few very dark municipalities far to the left of the bulk.

The department boxes overlap widely: the spread *within* departments is large compared with the differences *between* them. In 2012 Cochabamba alone runs from 0.039 to 0.563, the widest spread of any department. Pando and Chuquisaca have the highest medians in 2012 (0.336 and 0.317), Santa Cruz and Tarija the lowest (0.154 and 0.139). In 2024 every department is lower; Pando's median falls most, to 0.056, below those of Potosí (0.078) and Chuquisaca (0.074).

### 2.3 Population and poverty

Are the most populous municipalities less poor? A treemap (Plotly Express; the package has no such figure) sizes each municipality by its population, and `geo.scatter` sets poverty against population on a logarithmic axis.

In [12]:
d12 = df[df["year"] == 2012].assign(country="Bolivia")

px.treemap(d12, path=["country", "department", "name"], values="population", color="sdg1_2_mpi",
           labels=geo.labels(d12),
           title="Municipalities sized by population (2012 census), coloured by poverty")

In [13]:
pop24 = geo.scatter(df, "population", "sdg1_2_mpi", year=2024, log_x=True)
print(f"2024: slope {pop24.slope:.3f} per log point, R² {pop24.r2:.3f}")
geo.scatter(df, "population", "sdg1_2_mpi", year=2012, log_x=True, color="department")

2024: slope -0.009 per log point, R² 0.053


Across 343 units in 2012, **Multidimensional poverty index (0-1)** and **Population (census)** are negatively correlated (r = -0.34, a weak association): units with higher Population (census) tend to have lower Multidimensional poverty index (0-1).

The least-squares line has a slope of -0.0313: 1% more of Population (census) goes with 0.000313 lower Multidimensional poverty index (0-1), on average (Population (census) is in logs). The line accounts for 11% of the variation in Multidimensional poverty index (0-1) across units (R² = 0.11); the slope is statistically significant at the 1% level, by a test that treats the units as independent of each other.

_These are associations, not causal effects. A causal reading needs a research design._

In the treemap (click a department to zoom in) the largest blocks, the cities of Santa Cruz de la Sierra, El Alto, La Paz and Cochabamba, have the colours of low poverty; the poorest municipalities are small blocks.

The line confirms a negative but weak association: in 2012 each additional log point of population goes with an MPI 0.031 lower, and population accounts for only 11% of the variation. In 2024 the link is weaker still (slope −0.009, R² = 0.053). Small municipalities can be very poor or not poor at all.

### 2.4 Night-time lights and poverty

Do brighter municipalities have less poverty? This section uses the 339 units with lights. `geo.regression` puts two models side by side: the plain regression, and one with department fixed effects, whose slope compares municipalities of the same department.

In [14]:
geo.scatter(df, "ln_t400NTLpc", "sdg1_2_mpi", year=2012, color="department")

Across 339 units in 2012, **Multidimensional poverty index (0-1)** and **Log night-time lights per capita (trend)** are negatively correlated (r = -0.54, a moderate association): units with higher Log night-time lights per capita (trend) tend to have lower Multidimensional poverty index (0-1).

The least-squares line has a slope of -0.0335: one more unit of Log night-time lights per capita (trend) goes with 0.0335 lower Multidimensional poverty index (0-1), on average. The line accounts for 29% of the variation in Multidimensional poverty index (0-1) across units (R² = 0.29); the slope is statistically significant at the 1% level, by a test that treats the units as independent of each other.

_These are associations, not causal effects. A causal reading needs a research design._

In [15]:
ols24 = geo.regression(df, "sdg1_2_mpi", "ln_t400NTLpc", year=2024)
print(f"MPI 2024 on lights 2020: slope {ols24.df.loc[0, 'estimate']:.4f} "
      f"(SE {ols24.df.loc[0, 'std_error']:.4f}), R² {ols24.summary.loc[0, 'r2']:.3f}")

ols = geo.regression(df, "sdg1_2_mpi", [["ln_t400NTLpc"], ["ln_t400NTLpc"]], year=2012,
                     fixed_effects=[None, "department"])
ols.gt

MPI 2024 on lights 2020: slope -0.0180 (SE 0.0018), R² 0.255


GT(_tbl_data=                                       term        (1)        (2)
0  Log night-time lights per capita (trend)  -0.034***  -0.031***
1                                              (0.005)    (0.005)
2                                 Intercept   0.669***           
3                                              (0.063)           
4                  Department fixed effects         No        Yes
5                              Observations        339        339
6                                        R²      0.294      0.421
7                               Adjusted R²      0.292      0.405, _body=<great_tables._gt_data.Body object at 0x12d789090>, _boxhead=Boxhead([ColInfo(var='term', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None, stub_level=None), ColInfo(var='(1)', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None, stub_level=None), ColInfo(var='(2)', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12d78a610>, _spanners=Spanners([SpannerInfo(spanner_id='Multidimensional poverty index (0-1)', spanner_level=0, spanner_label='Multidimensional poverty index (0-1)', spanner_units=None, spanner_pattern=None, vars=['(1)', '(2)'], built=None)]), _heading=Heading(title='Regression of Multidimensional poverty index (0-1)', subtitle='Ordinary least squares, 2012', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12d788d50>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12d789a10>, _source_notes=['*** p<0.01, ** p<0.05, * p<0.10. Standard errors in parentheses: heteroskedasticity-robust (HC1).'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=[4], mask=None), grpname=None, colname='term', rownum=4, colnum=None, styles=[CellStyleBorders(sides='top', color='#7f7f7f', style='solid', weight='1px')]), StyleInfo(locname=LocBody(columns=None, rows=[4], mask=None), grpname=None, colname='(1)', rownum=4, colnum=None, styles=[CellStyleBorders(sides='top', color='#7f7f7f', style='solid', weight='1px')]), StyleInfo(locname=LocBody(columns=None, rows=[4], mask=None), grpname=None, colname='(2)', rownum=4, colnum=None, styles=[CellStyleBorders(sides='top', color='#7f7f7f', style='solid', weight='1px')])], _locale=<great_tables._gt_data.Locale object at 0x12d788810>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='tab

How different is the relationship from one department to another? The same scatter, department by department:

In [16]:
rows = []
for department, part in lit.groupby("department"):
    r12 = geo.scatter(part, "ln_t400NTLpc", "sdg1_2_mpi", year=2012)
    r24 = geo.scatter(part, "ln_t400NTLpc", "sdg1_2_mpi", year=2024)
    rows.append({"department": department, "units": r12.n, "slope 2012": r12.slope,
                 "correlation 2012": r12.corr, "correlation 2024 (lights 2020)": r24.corr})
print(pd.DataFrame(rows).sort_values("slope 2012").round(3).to_string(index=False))

l12 = lit[lit["year"] == 2012]
px.scatter(l12, x="ln_t400NTLpc", y="sdg1_2_mpi", facet_col="department", facet_col_wrap=3,
           trendline="ols", hover_name="name", labels=geo.labels(l12), height=700,
           title="Poverty and night-time lights by department, 2012")

department  units  slope 2012  correlation 2012  correlation 2024 (lights 2020)
Cochabamba     47      -0.085            -0.792                          -0.548
Chuquisaca     29      -0.068            -0.849                          -0.708
     Oruro     35      -0.045            -0.518                          -0.467
    Potosí     40      -0.027            -0.541                          -0.718
Santa Cruz     56      -0.025            -0.365                          -0.274
    La Paz     87      -0.024            -0.426                          -0.368
    Tarija     11      -0.015            -0.429                          -0.019
     Pando     15      -0.014            -0.321                          -0.329
      Beni     19      -0.013            -0.441                          -0.384


Night-time lights tell more about poverty than population does: on the 339 units with lights, one more log point of lights per capita goes with an MPI 0.034 lower, and lights alone account for 29% of the variation in poverty (R² = 0.294). The link persists in the latest period: poverty in 2024 on lights in 2020 gives a slope of −0.018 and R² = 0.255. With department fixed effects the slope is −0.031 (and R² rises to 0.42): almost all of the association holds between municipalities of the same department.

Within every department the 2012 slope is negative, but its size varies more than sixfold. It is steepest in Cochabamba (−0.085) and Chuquisaca (−0.068), where the correlations are also strongest (−0.79 and −0.85), and flattest in Beni (−0.013), Pando (−0.014) and Tarija (−0.015). Potosí's slope (−0.026) is flattened by Mojinete, the single point at the far left of its panel. In 2024 Chuquisaca and Potosí have the strongest correlations (both −0.71), and in Tarija the correlation vanishes (−0.02). Section 6 shows that this heterogeneity is geographic.

### 2.5 Infrastructure and connectivity correlates

Which infrastructure and connectivity indicators move with poverty? From the Atlas, in both periods:

| Variable | Meaning | Reference years |
| --- | --- | --- |
| `urbanization` | Urbanization rate (%) | 2024 only |
| `sdg7_1_cae` | Electricity coverage (% of population) | 2012, 2024 |
| `sdg9_8_tfc` | Fixed or mobile telephone (% of households) | 2012, 2024 |
| `sdg9_8_ci` | Internet access, fixed or mobile (% of households) | 2012, 2024 |
| `sdg9_8_drb` | Mobile base stations (per 1,000 people) | 2012, 2024 |
| `sdg11_2_atc` | Public-transport seats (per 1,000 people) | 2012, 2024 |
| `sdg8_10_dsb` | Financial access points (per 100,000 people) | 2012, 2024 |
| `sdg9_1_vpc` | Trunk roads crossing the municipality (number) | 2012, 2024 |

All are measured in the same year as the poverty index, except urbanization (2024 in both periods) and lights (2012 and 2020). `geo.correlation` uses the units with data for every variable, here the 339 with lights. The heatmap shows Spearman (rank) correlations, which are robust to the skewed indicators.

In [17]:
variables = ["sdg1_2_mpi", "ln_t400NTLpc", "ln_pop"] + infra
spearman12 = geo.correlation(df, variables, year=2012, method="spearman")
spearman12

Spearman correlations between 11 variables in 2012, on the 339 observations with data for all of them.

The strongest positive correlation is between **Electricity coverage (% of population)** and **Fixed or mobile telephone (% of households)** (r = 0.89): a strong association, in which units with more of one tend to have more of the other.

The strongest negative correlation is between **Multidimensional poverty index (0-1)** and **Electricity coverage (% of population)** (r = -0.90): a strong association, in which units with more of one tend to have less of the other.

The weakest correlation, in either direction, is between **Mobile base stations (per 1,000 people)** and **Public-transport seats (per 1,000 people)** (r = 0.04).

A coefficient near +1 or -1 means two variables move together closely, in the same or in opposite directions; near 0, that knowing one says little about the other. The table (`.gt`) sets Pearson correlations (above the diagonal) against Spearman rank correlations (below): a large gap between the two points to a non-linear relation or to outliers.

_These are associations, not causal effects. A causal reading needs a research design._

In [18]:
pearson12 = geo.correlation(df, variables, year=2012)
spearman24 = geo.correlation(df, variables, year=2024, method="spearman")
with_mpi = pd.DataFrame({"pearson 2012": pearson12.df["sdg1_2_mpi"],
                         "spearman 2012": spearman12.df["sdg1_2_mpi"],
                         "spearman 2024": spearman24.df["sdg1_2_mpi"]}).drop("sdg1_2_mpi")
with_mpi.insert(0, "label", with_mpi.index.map(labels))
print("Units:", spearman12.n, "| electricity and telephone, 2012:",
      round(spearman12.df.loc["sdg7_1_cae", "sdg9_8_tfc"], 2))
with_mpi.sort_values("spearman 2012").round(2)

Units: 339 | electricity and telephone, 2012: 0.89


,label,pearson 2012,spearman 2012,spearman 2024
sdg7_1_cae,Electricity coverage (% of population),-0.90,-0.90,-0.90
sdg9_8_tfc,Fixed or mobile telephone (% of households),-0.88,-0.89,-0.69
sdg9_8_ci,"Internet access, fixed or mobile (% of househo...",-0.58,-0.70,-0.69
ln_t400NTLpc,Log night-time lights per capita (trend),-0.54,-0.61,-0.58
urbanization,Urbanization rate (% of population),-0.57,-0.55,-0.48
sdg8_10_dsb,"Financial access points (per 100,000 people)",-0.47,-0.51,-0.52
sdg11_2_atc,"Public-transport seats (per 1,000 people)",-0.36,-0.48,-0.42
sdg9_1_vpc,Trunk roads crossing the municipality (number),-0.36,-0.37,-0.32
ln_pop,Log of population,-0.34,-0.32,-0.24
sdg9_8_drb,"Mobile base stations (per 1,000 people)",-0.18,-0.21,0.22


Every indicator correlates negatively with poverty in 2012: better-connected municipalities are less poor. The ranking:

- **Electricity coverage** (−0.90) and **fixed or mobile telephone** (−0.89), both from the 2012 census, are near mirror images of poverty. Part of this is mechanical, since census-based poverty measures count deprivations in household services. They are also strongly correlated with each other (0.89).
- **Internet access** (−0.70), **night-time lights** (−0.61), **urbanization** (−0.55, measured in 2024), **financial access points** (−0.51) and **public-transport seats** (−0.48) come next. For internet access and transit seats the Pearson correlations are weaker (−0.58 and −0.36): the relations are monotonic but far from linear.
- **Trunk roads** (−0.37), **population** (−0.32) and **mobile base stations** (−0.21) are the weakest.

In 2024 the picture is much the same: electricity coverage (−0.90) is still the closest correlate, while the telephone weakens (−0.69) and mobile base stations change sign (+0.22): by 2024 the poorer municipalities have more base stations per person.

To put these in context, here are the indicators of the whole Atlas most strongly associated with poverty (Spearman). Many of its indicators lack a value in some units, so each correlation uses the units that have both variables; that is plain pandas (`corrwith`), since `geo.correlation` keeps only the units with every variable. The list leaves out the goal indices and the overall index (composites that include poverty), the other poverty measure (unsatisfied basic needs, `sdg1_4_nbi`) and the indicators computed from the MPI itself (its gender, ethnic, migration and age parities and its Gini):

In [19]:
datasets = [d for d in geo.datasets()["dataset"] if d.startswith("sdg")]
atlas = geo.combine(datasets, scale="map343")
derived = ["sdg1_2_mpi", "sdg1_4_nbi", "sdg5_1_pgm", "sdg10_2_pel", "sdg10_2_mre", "sdg10_2_tpm",
           "sdg10_2_gip", "population", "urbanization", "year"]
candidates = [c for c in atlas.select_dtypes("number").columns
              if c not in derived and "index_" not in c and "rank_" not in c and "mun_id" not in c]

rank = pd.DataFrame({f"spearman {y}": atlas.loc[atlas["year"] == y, candidates].corrwith(
    atlas.loc[atlas["year"] == y, "sdg1_2_mpi"], method="spearman") for y in [2012, 2024]})
rank.insert(0, "label", rank.index.map(geo.labels(atlas)))
print("Indicators compared:", len(rank))
rank.reindex(rank["spearman 2012"].abs().sort_values(ascending=False).index).head(12).round(2)

Indicators compared: 74


,label,spearman 2012,spearman 2024
sdg7_1_cae,Electricity coverage (% of population),-0.90,-0.90
sdg9_8_tfc,Fixed or mobile telephone (% of households),-0.89,-0.69
sdg7_1_elc,Clean cooking energy (% of population),-0.85,-0.80
sdg11_1_ava,Adequate housing (% of dwellings),-0.84,-0.86
sdg6_1_cap,Drinking-water coverage (% of population),-0.73,-0.73
sdg9_8_ci,"Internet access, fixed or mobile (% of househo...",-0.71,-0.69
sdg1_4_eep,Energy poverty (% of dwellings),0.70,0.59
sdg1_4_ssb,Access to the 3 basic services (% of households),-0.70,-0.72
sdg7_1_cep,Residential electricity consumption (kWh per p...,-0.69,-0.44
sdg13_1_icc,Climate-change vulnerability index (0-3),0.60,0.48


Household energy and services dominate in both periods: electricity coverage, telephones, clean cooking energy, adequate housing, drinking water, internet and access to the three basic services all have Spearman correlations between −0.70 and −0.90 in 2012, and all but the telephone stay close to them in 2024. Energy poverty (+0.70) and climate-change vulnerability (+0.60) rise with poverty, as does the secondary-school dropout rate of girls (+0.59). Residential electricity consumption (−0.69) and employment in tourism-related activities (−0.60) are higher in the less poor municipalities; the link with electricity consumption weakens to −0.44 in 2024.

## 3. Maps

With a list of years, `geo.map` draws both periods in the same classes, so the colours are comparable. Fisher–Jenks natural breaks group similar values; the figure keeps the class limits, which gives the number of units in each class. The box-plot rule (values beyond 1.5 times the interquartile range from the quartiles) flags the outliers.

In [20]:
fig = geo.map(df, "sdg1_2_mpi", year=[2012, 2024], scheme="fisher_jenks")
bins = fig.layout.meta["bins"]
classes = pd.cut(df["sdg1_2_mpi"], bins, include_lowest=True)
print(pd.crosstab(classes, df["year"]))
high12 = df[(df["year"] == 2012) & (df["sdg1_2_mpi"] > bins[3])]
high24 = df[(df["year"] == 2024) & (df["sdg1_2_mpi"] > bins[2])]
print(f"2012, above {bins[3]}:", high12["department"].value_counts().to_dict())
print(f"2024, above {bins[2]}:", high24["department"].value_counts().to_dict())

q = geo.describe(df, "sdg1_2_mpi").set_index("year")
for y in [2012, 2024]:
    low = q.loc[y, "p25"] - 1.5 * (q.loc[y, "p75"] - q.loc[y, "p25"])
    high = q.loc[y, "p75"] + 1.5 * (q.loc[y, "p75"] - q.loc[y, "p25"])
    out = df[(df["year"] == y) & ((df["sdg1_2_mpi"] > high) | (df["sdg1_2_mpi"] < low))]
    print(f"{y}: box-plot fences {low:.3f} and {high:.3f}; outliers:",
          ", ".join(f"{n} ({d})" for n, d in zip(out["name"], out["department"])))
fig

year            2012  2024
sdg1_2_mpi                
(0.002, 0.084]    20   261
(0.084, 0.172]    87    73
(0.172, 0.273]   123     8
(0.273, 0.386]    87     1
(0.386, 0.563]    26     0
2012, above 0.273: {'La Paz': 24, 'Chuquisaca': 21, 'Potosí': 21, 'Cochabamba': 14, 'Oruro': 12, 'Pando': 11, 'Beni': 7, 'Santa Cruz': 3}
2024, above 0.172: {'Chuquisaca': 3, 'Cochabamba': 2, 'Potosí': 2, 'Oruro': 1, 'Pando': 1}
2012: box-plot fences -0.077 and 0.537; outliers: Poroma (Chuquisaca), TIOC-Raqaypampa (Cochabamba)
2024: box-plot fences -0.050 and 0.163; outliers: Azurduy (Chuquisaca), Incahuasi (Chuquisaca), Poroma (Chuquisaca), Vila Vila (Cochabamba), TIOC-Raqaypampa (Cochabamba), TIOC-Territorio Indígena Multiétnico (Beni), El Choro (Oruro), San Pedro (Pando), San Pedro (Potosí), Toro Toro (Potosí), Tacobamba (Potosí), Ckochas (Potosí)


In [21]:
geo.map(df, "ln_t400NTLpc", year=2012, scheme="quantiles", map="map339")

The box-plot rule finds no low outliers (the lower fence is below zero) and two high outliers above 0.537 in 2012: TIOC-Raqaypampa (Cochabamba) and Poroma (Chuquisaca). In 2024 it finds twelve, above 0.163, seven of them in Potosí and Chuquisaca.

In the common classes, the two periods hardly overlap. In 2012 only 20 units are in the lowest class (an MPI of up to 0.084) and 113 are in the two highest (above 0.273). In 2024, 261 units are in the lowest class and one is above 0.273.

Poverty is not scattered at random. The 113 units of the two highest classes of 2012 include 21 of Chuquisaca's 29 municipalities, 11 of Pando's 15 and half of Potosí's (21 of 42), with 24 in La Paz, 14 in Cochabamba and 12 in Oruro; Santa Cruz has three and Tarija none. In 2024 only nine units are above 0.172: three in Chuquisaca, two each in Cochabamba and Potosí, and one each in Oruro and Pando. The lights map shows roughly the reverse of the 2012 poverty map (the rank correlation of the two variables is −0.60, section 2.5). The next section measures these patterns formally.

## 4. Spatial dependence

- To what extent is a municipality's poverty similar to that of its neighbours?
- Is there an overall pattern of clustering?
- Where are the statistically significant clusters and spatial outliers?
- Do poor municipalities sit next to dark (or bright) ones?

### 4.1 Spatial weights

The spatial functions build queen contiguity weights by default. This notebook keeps the choice of the original: each municipality's neighbours are its **6 nearest municipalities** (by centroid). *k*-nearest neighbours give every unit the same number of neighbours and no islands; the weights are row-standardized, so the spatial lag is the neighbours' average. `geo.weights` builds them, and they are passed as `w=` to every function below. A second set, `w_lit`, is built the same way on the 339 units with lights, on the 2015 map (`map339`) that their values belong to.

In [22]:
w = geo.weights(df, method="knn", k=6)
w_lit = geo.weights(lit, method="knn", k=6, map="map339")   # lights belong to the 2015 map
print(w.n, "and", w_lit.n, "units")
print(geo.connectivity(df, w).interpret())

343 and 339 units


The spatial weights graph (**6-nearest-neighbor (metric centroids), row-standardized, n=343**) links 343 units with 6 neighbors each on average (ranging from 6 to 6); 1.75% of all possible unit pairs are connected, so each spatial lag averages over a small, local set of neighbors.

The graph forms a **single connected component** — every unit can reach every other through a chain of neighbors, which is what global spatial statistics (Moran's I, spatial models) implicitly assume.

No unit is an island: every unit has at least one neighbor, so spatial lags are defined everywhere.

_These are associations, not causal effects. A causal reading needs a research design._


### 4.2 Global spatial autocorrelation

In [23]:
moran12 = geo.moran(df, "sdg1_2_mpi", year=2012, w=w)
moran12

Global Moran's I for **Multidimensional poverty index (0-1)** in 2012 is 0.448, against an expectation of -0.00292 under spatial randomness — statistically significant at the 1% level (pseudo p = 0.001 from 999 permutations, under 6-nearest-neighbor (metric centroids), row-standardized, n=343).

The dependence is **positive**: similar values cluster in space — high values sit next to high values and low next to low, so the map shows contiguous patches rather than a random scatter.

245 of 343 units (71%) fall in the clustering quadrants of the scatter (High-High or Low-Low); the rest are surrounded by neighbors unlike themselves. The slope of the fitted line equals Moran's I under row-standardized weights, so a steeper line means stronger clustering.

_These are associations, not causal effects. A causal reading needs a research design._

In [24]:
tests = [("sdg1_2_mpi", 2012, df, w, "map343"), ("sdg1_2_mpi", 2024, df, w, "map343"),
         ("ln_pop", 2012, df, w, "map343"), ("urbanization", 2024, df, w, "map343"),
         ("sdg9_8_tfc", 2012, df, w, "map343"), ("sdg7_1_cae", 2012, df, w, "map343"),
         ("ln_t400NTLpc", 2012, lit, w_lit, "map339")]
rows = []
for variable, year, data, weights, on in tests:
    m = geo.moran(data, variable, year=year, w=weights, map=on)
    rows.append({"variable": labels[variable], "year": year, "units": len(m.df),
                 "Moran's I": m.moran_i, "z": m.z_sim, "pseudo p": m.p_sim})
pd.DataFrame(rows).round(3)

,variable,year,units,Moran's I,z,pseudo p
0,Multidimensional poverty index (0-1),2012,343,0.448,15.026,0.001
1,Multidimensional poverty index (0-1),2024,343,0.388,13.001,0.001
2,Log of population,2012,343,0.349,12.467,0.001
3,Urbanization rate (% of population),2024,343,0.290,9.986,0.001
4,Fixed or mobile telephone (% of households),2012,343,0.409,14.168,0.001
5,Electricity coverage (% of population),2012,343,0.414,14.335,0.001
6,Log night-time lights per capita (trend),2012,339,0.324,11.485,0.001


Poverty is strongly and significantly clustered: in 2012 Moran's I is 0.45, far above its expected value under spatial randomness (−0.003), with a pseudo p-value of 0.001 (the smallest possible with 999 permutations). It is the most clustered of the variables. In 2024 the clustering is a little weaker (0.39) but still strong. Electricity coverage (0.41), telephones (0.41), log population (0.35), night-time lights (0.32, on the 339 units of the 2015 map) and urbanization (0.29) are also significantly clustered. In the scatterplot most municipalities fall in the High-High and Low-Low quadrants: poor municipalities have poor neighbours, and less poor ones less poor neighbours.

### 4.3 Local spatial autocorrelation (LISA)

The local Moran's I splits the global statistic into one value per municipality. With 999 permutations and a 10% significance level (as in the original; the default of `geo.lisa` is 5%):

- **High-High**: poor municipalities surrounded by poor neighbours (poverty hot spots)
- **Low-Low**: less poor municipalities surrounded by less poor neighbours (cold spots)
- **High-Low** / **Low-High**: spatial outliers

In [25]:
lisa12 = geo.lisa(df, "sdg1_2_mpi", year=2012, w=w, alpha=0.10)
lisa12

Local Moran statistics (LISA) locate *where* **Multidimensional poverty index (0-1)** in 2012 clusters or stands out, under 6-nearest-neighbor (metric centroids), row-standardized, n=343. The accompanying global Moran's I is 0.448 (pseudo p = 0.001), consistent with overall clustering of similar values.

At the 0.1 significance level, 157 of 343 units show significant local association: **52 High-High** hot spots (high values surrounded by high neighbors) and **83 Low-Low** cold spots (low surrounded by low) mark clustering, while **10 High-Low** and **12 Low-High** units are spatial outliers that break with their surroundings. The remaining 186 units are not significant — their local pattern is compatible with randomness.

LISA pseudo p-values are computed unit by unit without a multiple-testing adjustment, so treat borderline clusters cautiously and read the map as descriptive of where dependence concentrates.

_These are associations, not causal effects. A causal reading needs a research design._

In [26]:
local12 = geo.add_names(lisa12.df, groups=["department"]).merge(
    df.loc[df["year"] == 2012, ["geo_id", "sdg1_2_mpi"]], on="geo_id")
for c in ["High-High", "Low-Low", "High-Low", "Low-High"]:
    sel = local12[local12["cluster"] == c].sort_values(
        "sdg1_2_mpi", ascending=c in ["Low-Low", "Low-High"]).head(8)
    print(f"{c}:", ", ".join(f"{n} ({d})" for n, d in zip(sel["name"], sel["department"])))
pd.crosstab(local12["department"], local12["cluster"], margins=True)

High-High: TIOC-Raqaypampa (Cochabamba), Poroma (Chuquisaca), Azurduy (Chuquisaca), Sena (Pando), San Pedro (Potosí), Ckochas (Potosí), San Pedro (Pando), Tinguipaya (Potosí)
Low-Low: Nuestra Señora de La Paz (La Paz), Colcapirhua (Cochabamba), Tarija (Tarija), Santa Cruz de la Sierra (Santa Cruz), Bermejo (Tarija), El Alto (La Paz), Cliza (Cochabamba), Punata (Cochabamba)
High-Low: El Choro (Oruro), Morochata (Cochabamba), La Asunta (La Paz), Mojinete (Potosí), Postrervalle (Santa Cruz), Tito Yupanqui (La Paz), San Miguel de Velasco (Santa Cruz), San Rafael (Santa Cruz)
Low-High: Sucre (Chuquisaca), Guayaramerín (Beni), Porvenir (Pando), Santa Ana (Beni), Curahuara de Carangas (Oruro), Camargo (Chuquisaca), Monteagudo (Chuquisaca), Colquiri (La Paz)


cluster,High-High,High-Low,Low-High,Low-Low,Not significant,All
department,,,,,,
Beni,2,0,2,0,16,20
Chuquisaca,15,0,3,1,10,29
Cochabamba,8,1,0,21,18,48
La Paz,5,2,3,19,58,87
Oruro,2,1,1,3,28,35
Pando,8,0,3,0,4,15
Potosí,12,1,0,0,29,42
Santa Cruz,0,5,0,31,20,56
Tarija,0,0,0,8,3,11


How persistent are the clusters? `geo.lisa_transitions` runs the same analysis in both periods and counts the moves between types. Its map shows the types of 2024, with the units that changed type outlined.

In [27]:
trans = geo.lisa_transitions(df, "sdg1_2_mpi", w=w, alpha=0.10)
trans

Local Moran statistics (LISA) of **Multidimensional poverty index (0-1)** were computed in 2012 and in 2024 for the same 343 units, under 6-nearest-neighbor (metric centroids), row-standardized, n=343, and each unit was assigned to a type at the 0.1 level. Global Moran's I is 0.448 in 2012 (pseudo p = 0.001) and 0.388 in 2024 (pseudo p = 0.001): the overall clustering of similar values weakened.

**259 of 343 units (76%) are of the same type in both years** and 84 changed type. Of the 157 units in a significant cluster or outlier in 2012, 102 kept their type, 43 are no longer significant in 2024 and 12 switched to another type. 29 units that were not significant in 2012 belong to a cluster or are an outlier in 2024.

Of the 52 **High-High** hot spots (high values surrounded by high values) of 2012, 32 are still High-High in 2024, a year with 36 of them. Of the 83 **Low-Low** cold spots (low values surrounded by low values) of 2012, 63 are still Low-Low in 2024, a year with 85 of them. The most common change is from Low-Low to Not significant (18 units).

Read the changes with care. A unit's type depends on whether its pseudo p-value is below the threshold, computed unit by unit without a multiple-testing adjustment, so moves to and from Not significant can come from borderline p-values. High and low are relative to the mean of each year, and a unit can change type because its neighbours' values changed while its own did not.

_These are associations, not causal effects. A causal reading needs a research design._

In [28]:
trans.fig_map.show()
types = geo.add_names(trans.df, groups=["department"])
pd.crosstab(types["department"], types["cluster_2024"], margins=True)

cluster_2024,High-High,High-Low,Low-High,Low-Low,Not significant,All
department,,,,,,
Beni,3,0,1,0,16,20
Chuquisaca,9,1,3,3,13,29
Cochabamba,5,2,1,12,28,48
La Paz,1,2,3,16,65,87
Oruro,4,4,0,2,25,35
Pando,2,0,3,0,10,15
Potosí,12,0,2,2,26,42
Santa Cruz,0,0,0,41,15,56
Tarija,0,0,0,9,2,11


Of the 343 municipalities, 157 are significant at 10% in 2012: 52 High-High, 83 Low-Low, 10 High-Low and 12 Low-High.

- **Hot spots (High-High)** form two blocks. In the south, the hot spots are rural municipalities of Chuquisaca (15), Potosí (12) and Cochabamba (8), with the poorest municipalities of the country: Raqaypampa, Poroma, Azurduy, San Pedro, Ckochas, Tinguipaya. In the north, 8 of Pando's 15 municipalities are hot spots, including Sena and San Pedro.
- **Cold spots (Low-Low)** are the lowlands of Santa Cruz (31), the Cochabamba valley (21), the La Paz–El Alto area (19) and Tarija (8).
- **Low-High outliers** are less poor towns in poor surroundings, mostly urban centres: Sucre, Guayaramerín, Porvenir, Camargo and Monteagudo.
- **High-Low outliers** are poor municipalities in less poor surroundings, for instance El Choro (Oruro), Morochata (Cochabamba), La Asunta (La Paz), Mojinete (Potosí) and San Miguel de Velasco and San Rafael (Santa Cruz).

The clusters are persistent: 259 of the 343 units (76%) are of the same type in both periods. In 2024, 143 municipalities are significant (36 High-High, 85 Low-Low, 9 High-Low, 13 Low-High); 32 of the 52 hot spots of 2012 are still hot spots and 63 of the 83 cold spots still cold spots. The hot spots retreat to the Andean core (Potosí 12, Chuquisaca 9, Cochabamba 5, Oruro 4) and Pando keeps only two, while the Santa Cruz cold spot grows from 31 to 41 municipalities. Most of the other changes are to or from "Not significant", a category that depends on a p-value threshold.

### 4.4 Bivariate spatial association: poverty and neighbours' lights

The bivariate Moran's I relates a municipality's poverty to the **average night-time lights of its neighbours**. Its local version classifies each municipality by its own poverty (first word) and its neighbours' lights (second word): *High-Low* means a poor municipality in a dark neighbourhood, *Low-High* a less poor one in a bright neighbourhood.

The package has no bivariate Moran functions, so this section calls the esda library directly, on the 339 units with lights. The weights are `w_lit` from `geo.weights`; the values must be in the order of the weights (`w_lit.id_order`).

In [29]:
quadrant = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}

def bivariate(year):
    """Bivariate Moran's I and local types of poverty against the neighbours' lights."""
    d = lit[lit["year"] == year].set_index("geo_id").loc[w_lit.id_order].reset_index()
    x, y = d["sdg1_2_mpi"].to_numpy(), d["ln_t400NTLpc"].to_numpy()
    np.random.seed(12345)
    bv = Moran_BV(x, y, w_lit, permutations=999)
    local = Moran_Local_BV(x, y, w_lit, permutations=999, seed=12345)
    d["poverty (standardized)"] = (x - x.mean()) / x.std()
    d["neighbours' lights (standardized)"] = lag_spatial(w_lit, (y - y.mean()) / y.std())
    d["cluster"] = pd.Series(local.q).map(quadrant).where(local.p_sim < 0.10, "Not significant")
    return bv, d

bv12, d12 = bivariate(2012)
bv24, d24 = bivariate(2024)
print(f"Bivariate Moran's I, poverty 2012 and neighbours' lights 2012: {bv12.I:.3f} "
      f"(pseudo p = {bv12.p_sim:.3f}); correlation with own lights: "
      f"{d12['sdg1_2_mpi'].corr(d12['ln_t400NTLpc']):.3f}")
print(f"Bivariate Moran's I, poverty 2024 and neighbours' lights 2020: {bv24.I:.3f} "
      f"(pseudo p = {bv24.p_sim:.3f})")

colors = {"High-High": "#d7191c", "Low-Low": "#2c7bb6", "Low-High": "#abd9e9",
          "High-Low": "#fdae61", "Not significant": "#d3d3d3"}       # as in the package's LISA maps
px.scatter(d12, x="poverty (standardized)", y="neighbours' lights (standardized)", color="cluster",
           color_discrete_map=colors, hover_name="name", trendline="ols", trendline_scope="overall",
           title="Bivariate Moran scatterplot: poverty and neighbours' lights, 2012")

Bivariate Moran's I, poverty 2012 and neighbours' lights 2012: -0.269 (pseudo p = 0.001); correlation with own lights: -0.542
Bivariate Moran's I, poverty 2024 and neighbours' lights 2020: -0.291 (pseudo p = 0.001)


In [30]:
gdf = geo.boundaries("map339").merge(d12[["geo_id", "department", "cluster"]], on="geo_id")
gdf["geometry"] = gdf.simplify(0.02)                 # lighter boundaries keep the figure small

px.choropleth_map(gdf, geojson=gdf.geometry, locations=gdf.index, color="cluster",
                  color_discrete_map=colors, hover_name="name",
                  center={"lat": -16.3, "lon": -64}, zoom=4, height=600,
                  title="Bivariate LISA: own poverty and neighbours' lights, 2012").show()
pd.crosstab(d12["department"], d12["cluster"], margins=True)

cluster,High-High,High-Low,Low-High,Low-Low,Not significant,All
department,,,,,,
Beni,0,4,0,6,9,19
Chuquisaca,2,4,0,1,22,29
Cochabamba,5,1,20,0,21,47
La Paz,0,22,5,16,44,87
Oruro,1,0,2,0,32,35
Pando,0,5,0,2,8,15
Potosí,0,11,1,4,24,40
Santa Cruz,3,0,31,0,22,56
Tarija,0,0,8,0,3,11


In [31]:
for c in ["Low-High", "High-Low", "Low-Low", "High-High"]:
    sel = d12[d12["cluster"] == c]
    units = zip(sel["name"], sel["department"])
    print(f"{c} ({len(sel)}):", ", ".join(f"{n} ({d})" for n, d in units))

# The poverty clusters of section 4.3 (rows) against the bivariate types (columns), 339 units
both = d12[["geo_id", "cluster"]].merge(lisa12.df[["geo_id", "cluster"]], on="geo_id",
                                        suffixes=("_bivariate", "_poverty"))
pd.crosstab(both["cluster_poverty"], both["cluster_bivariate"], margins=True)

Low-High (67): Arani (Cochabamba), Santiváñez (Cochabamba), Cochabamba (Cochabamba), Colomi (Cochabamba), Sacaba (Cochabamba), Arbieto (Cochabamba), Tarata (Cochabamba), Cliza (Cochabamba), Toco (Cochabamba), Tolata (Cochabamba), Punata (Cochabamba), San Benito (Cochabamba), Tacachi (Cochabamba), Villa Rivero (Cochabamba), Colcapirhua (Cochabamba), Quillacollo (Cochabamba), Sipesipe (Cochabamba), Tiquipaya (Cochabamba), Vinto (Cochabamba), Tiraque (Cochabamba), Calamarca (La Paz), Pucarani (La Paz), Achocalla (La Paz), El Alto (La Paz), Mecapaca (La Paz), Pazña (Oruro), Poopó (Oruro), Tahua (Potosí), Cotoca (Santa Cruz), El Torno (Santa Cruz), La Guardia (Santa Cruz), Porongo (Santa Cruz), Santa Cruz de la Sierra (Santa Cruz), Pailón (Santa Cruz), Roboré (Santa Cruz), Boyuibe (Santa Cruz), Cabezas (Santa Cruz), Camiri (Santa Cruz), El Carmen Rivero Tórrez (Santa Cruz), Puerto Quijarro (Santa Cruz), Puerto Suárez (Santa Cruz), Buena Vista (Santa Cruz), San Carlos (Santa Cruz), San Juan 

cluster_bivariate,High-High,High-Low,Low-High,Low-Low,Not significant,All
cluster_poverty,,,,,,
High-High,0,22,0,0,27,49
High-Low,4,1,0,0,5,10
Low-High,0,0,0,5,7,12
Low-Low,1,0,55,3,24,83
Not significant,6,24,12,21,122,185
All,11,47,67,29,185,339


The bivariate Moran's I is −0.27 (pseudo p = 0.001): poor municipalities tend to have dark neighbours. The association is weaker than with a municipality's own lights (correlation −0.54), as expected, but clearly significant. It is slightly stronger between poverty in 2024 and lights in 2020 (−0.291, pseudo p = 0.001).

The bivariate LISA flags 154 of the 339 municipalities at 10%, dominated by the two "expected" combinations:

- **Low-High** (67): less poor municipalities in bright neighbourhoods, in Santa Cruz (31), the Cochabamba valley (20), Tarija (8) and around El Alto (La Paz, 5).
- **High-Low** (47): poor municipalities in dark neighbourhoods, mostly in La Paz (22, from Ixiamas and Apolo in the north to Charazani and Pelechuco), Potosí (11, Colquechaca among them), Pando (5), Beni (4) and Chuquisaca (4).
- **Low-Low** (29): less poor municipalities with dark neighbours, mainly in La Paz (16; the Yungas towns of Caranavi, Chulumani and Irupana), Beni (6; Trinidad, Rurrenabaque) and Potosí (4; Tupiza, Villazón).
- **High-High** (11): poor municipalities with bright neighbours, for example Villa Tunari in the Chapare (Cochabamba), and Charagua, Cuevo and San Matías (Santa Cruz).

The cross-tabulation links the two analyses. Of the 49 poverty hot spots that have lights, 22 are poor municipalities in dark neighbourhoods; of the 83 cold spots, 55 sit in bright neighbourhoods. Most of the other hot and cold spots (27 and 24) are not significant in the bivariate map.

## 5. Spatial inequality

How unequal is poverty across municipalities, and how much of it lies between departments?

`geo.inequality` gives the Gini and Theil indices in each period. With weights, it adds the spatial Gini (Rey and Smith, 2013), which splits the Gini's sum of absolute differences into neighbour pairs and all other pairs: if neighbours are more alike than random pairs, the neighbour pairs contribute less than their share of all pairs. The p-value comes from 999 random permutations of the values over the map. `geo.theil_decomposition` splits the Theil index exactly into inequality *between* department means and *within* departments.

In [32]:
ineq = geo.inequality(df, "sdg1_2_mpi", w=w, permutations=999)
ineq

Cross-sectional inequality in **Multidimensional poverty index (0-1)** is tracked across 343 units over 2 years (2012 to 2024).

The Gini index rose from 0.254 to 0.367 between the first and last year.

The Theil index rose from 0.104 to 0.219 between the first and last year.

With two years (2012 and 2024) the measures can be compared, but their trend cannot be tested: that needs three or more years.

In the latest year, inequality between *neighboring* units under the weights (6-nearest-neighbor (metric centroids), row-standardized, n=343) contributes 0.00471 to the Gini — about 1% of the overall Gini; the rest comes from pairs that are not neighbors.

The permutation test (p = 0.001) indicates the non-neighbor component of inequality is larger than expected under spatial randomness — differences line up with geography rather than being scattered among neighbors.

On the Lorenz curve of 2024, the half of the units with the lowest values holds 25% of the total (32% in 2012) and the tenth with the highest values holds 25% (19%); with no inequality they would hold 50% and 10%.

_These are associations, not causal effects. A causal reading needs a research design._

In [33]:
def neighbour_share(result, w):
    """The Gini owed to neighbour pairs, in % of the Gini, against their share of all pairs."""
    table = result.df[["year", "gini", "theil", "gini_spatial", "gini_spatial_p"]].copy()
    table["Gini from neighbour pairs (%)"] = 100 * table["gini_spatial"] / table["gini"]
    table["neighbour pairs (% of all pairs)"] = 100 * w.mean_neighbors / (w.n - 1)
    return table.round(3)

neighbour_share(ineq, w)

,year,gini,theil,gini_spatial,gini_spatial_p,Gini from neighbour pairs (%),neighbour pairs (% of all pairs)
0,2012,0.254,0.104,0.003,0.001,1.221,1.754
1,2024,0.367,0.219,0.005,0.001,1.284,1.754


In [34]:
geo.theil_decomposition(df, "sdg1_2_mpi", by="department", permutations=999)

The Theil index of **Multidimensional poverty index (0-1)** splits additively into inequality **between** the 9 department groups and inequality **within** them (between + within = total, exactly).

In the latest year (2024), the total Theil index is 0.219: about 21% of it lies between department groups and 79% within them — most inequality plays out among units inside the same department.

Over the window, the between-group share stayed at about 21%.

The permutation test on the between component (p = 0.001) indicates the observed grouping captures more inequality than random reassignments of units to groups typically do.

_These are associations, not causal effects. A causal reading needs a research design._

The Gini and Theil indices need non-negative levels, so night-time lights enter here as **lights per capita in levels**, exp(`ln_t400NTLpc`), not in logs. The lights have a value in every year from 2012 to 2020, so here the measures have a trend, which `geo.inequality` tests.

In [35]:
lights = geo.load("ntl", variables="ln_t400NTLpc")
lights["ntl_pc"] = np.exp(lights["ln_t400NTLpc"])

ineq_lights = geo.inequality(lights, "ntl_pc", w=w_lit, permutations=999)
print(ineq_lights.interpret())
neighbour_share(ineq_lights, w_lit).iloc[[0, -1]]

Cross-sectional inequality in **ntl_pc** is tracked across 339 units over 9 years (2012 to 2020).

The Gini index fell from 0.634 to 0.568 between the first and last year.

The Theil index fell from 0.77 to 0.681 between the first and last year.

The log-trend of the Gini index is -0.0152 per year (statistically significant at the 1% level), so inequality by this measure is **narrowing**.

The log-trend of the Theil index is -0.0227 per year (not statistically significant at conventional levels), so the movement is not distinguishable from a flat trend.

In the latest year, inequality between *neighboring* units under the weights (6-nearest-neighbor (metric centroids), row-standardized, n=339) contributes 0.00815 to the Gini — about 1% of the overall Gini; the rest comes from pairs that are not neighbors.

The permutation test (p = 0.001) indicates the non-neighbor component of inequality is larger than expected under spatial randomness — differences line up with geography rather than 

,year,gini,theil,gini_spatial,gini_spatial_p,Gini from neighbour pairs (%),neighbour pairs (% of all pairs)
0,2012,0.634,0.770,0.008,0.001,1.321,1.775
8,2020,0.568,0.681,0.008,0.001,1.436,1.775


In [36]:
geo.theil_decomposition(lights, "ntl_pc", by="department").df.iloc[[0, -1]].round(3)

,year,theil,between,within,between_share
0,2012,0.770,0.169,0.601,0.219
8,2020,0.681,0.077,0.604,0.113


Municipal poverty in 2012 has a Gini of 0.25 and a Theil of 0.10. Only 21% of the Theil index is inequality *between* departments; 79% is *within* them. This matches the overlapping boxes of section 2.2: departments differ, but most of the inequality lies between municipalities of the same department. The between part is still larger than random groupings of municipalities would give (pseudo p = 0.001).

As poverty fell, its *relative* inequality rose: in 2024 the Gini is 0.37 and the Theil 0.22, with the same 21/79 split. Absolute gaps narrowed (the standard deviation fell from 0.106 to 0.044), but the least poor municipalities cut poverty proportionally more than the poorest (section 2.1). On the Lorenz curve, the tenth of the units with the highest values holds 25% of the total in 2024, against 19% in 2012.

Night-time lights per capita are far more unequal (Gini 0.63 and Theil 0.77 in 2012), with a similar split in 2012 (22% between departments). Their Gini fell to 0.57 by 2020, a significant downward trend (that of the Theil index is not significant), and the part between departments fell to 11%.

The spatial Gini shows the clustering from another angle. Neighbour pairs are 1.75% of all pairs (1.78% among the 339 units with lights) but contribute only 1.22% of the Gini of poverty in 2012, 1.28% in 2024 and 1.32% of that of lights in 2012. In all three cases the permutation p-value is 0.001: neighbouring municipalities are much more alike than random pairs.

## 6. Spatial heterogeneity: multiscale geographically weighted regression

The regression of section 2.4 imposes one slope on all municipalities, and section 2.4 also showed that the slope differs between departments. A geographically weighted regression fits a regression around every municipality, weighting the others by distance, so the slope becomes a surface that can be mapped. `geo.mgwr` fits the multiscale version (MGWR), which finds one **bandwidth** per term: the number of nearest municipalities that enter each local regression. A bandwidth close to the number of units means a relationship that is about the same everywhere; a small one, a relationship that changes over short distances.

The function follows the mgwr library: the variables are z-standardized, so the coefficients are in standard deviations, and the local t-tests are corrected for multiple testing. The sample is the 339 units with lights, on the 2015 map (`map="map339"`), and one fit takes about half a minute.

In [37]:
mg = geo.mgwr(df, "sdg1_2_mpi", "ln_t400NTLpc", year=2012, map="map339")
print(mg.notes[0])
mg.gt_bw

4 of 343 units are left out: no data for sdg1_2_mpi, ln_t400NTLpc in 2012, or no boundary


GT(_tbl_data=                                       Term Bandwidth 95% interval    ENP Corrected alpha Critical |t|
0                                 Intercept        44     44 to 48  20.93        0.002389         3.06
1  Log night-time lights per capita (trend)        65     64 to 85  13.47        0.003712        2.922, _body=<great_tables._gt_data.Body object at 0x12f2af6d0>, _boxhead=Boxhead([ColInfo(var='Term', type=<ColInfoTypeEnum.stub: 2>, column_label='Term', column_align='left', column_width=None, stub_level=0), ColInfo(var='Bandwidth', type=<ColInfoTypeEnum.default: 1>, column_label='Bandwidth', column_align='right', column_width=None, stub_level=None), ColInfo(var='95% interval', type=<ColInfoTypeEnum.default: 1>, column_label='95% interval', column_align='left', column_width=None, stub_level=None), ColInfo(var='ENP', type=<ColInfoTypeEnum.default: 1>, column_label='ENP', column_align='right', column_width=None, stub_level=None), ColInfo(var='Corrected alpha', type=<ColInfoTypeEnum.default: 1>, column_label='Corrected alpha', column_align='right', column_width=None, stub_level=None), ColInfo(var='Critical |t|', type=<ColInfoTypeEnum.default: 1>, column_label='Critical |t|', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12f2ad990>, _spanners=Spanners([]), _heading=Heading(title='MGWR bandwidths', subtitle='adaptive nearest-neighbor bandwidths out of 339 units', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12e702350>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12e703950>, _source_notes=['Smaller bandwidths mean the association varies over shorter distances (a more local process); a bandwidth near n is effectively global.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12e701810>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), tabl

In [38]:
mg.figs["ln_t400NTLpc"]

In [39]:
slope = "ln_t400NTLpc_coef"
local = geo.add_names(mg.df, groups=["department"])
print(f"R²: global {ols.summary.loc[0, 'r2']:.3f}, MGWR {mg.r2:.3f}")
print(f"Local slopes: {local[slope].min():.2f} to {local[slope].max():.2f}; negative: "
      f"{(local[slope] < 0).sum()} of {len(local)}; significant after correction: "
      f"{local['ln_t400NTLpc_significant'].sum()}")
print("Largest local condition number:", round(local["local_cn"].max(), 1))
local.groupby("department", as_index=False).agg(
    units=("geo_id", "size"), mean_slope=(slope, "mean"),
    significant=("ln_t400NTLpc_significant", "sum")).sort_values("mean_slope").round(2)

R²: global 0.294, MGWR 0.647
Local slopes: -1.35 to -0.17; negative: 339 of 339; significant after correction: 260
Largest local condition number: 4.9


,department,units,mean_slope,significant
2,Cochabamba,47,-1.23,47
1,Chuquisaca,29,-1.02,29
6,Potosí,40,-0.92,35
4,Oruro,35,-0.70,26
8,Tarija,11,-0.57,11
7,Santa Cruz,56,-0.47,21
3,La Paz,87,-0.41,74
0,Beni,19,-0.29,6
5,Pando,15,-0.28,11


The same model for the latest period, poverty in 2024 on lights in 2020:

In [40]:
mg24 = geo.mgwr(df, "sdg1_2_mpi", "ln_t400NTLpc", year=2024, map="map339")
print("Bandwidths:", mg24.bw, "| 95% intervals:", mg24.bw_interval)
print(f"R²: global {ols24.summary.loc[0, 'r2']:.3f}, MGWR {mg24.r2:.3f}; negative slopes: "
      f"{(mg24.df[slope] < 0).sum()} of {len(mg24.df)}; significant: "
      f"{mg24.df['ln_t400NTLpc_significant'].sum()}")

Bandwidths: {'const': 44.0, 'ln_t400NTLpc': 101.0} | 95% intervals: {'const': (44.0, 48.0), 'ln_t400NTLpc': (101.0, 112.0)}
R²: global 0.255, MGWR 0.541; negative slopes: 339 of 339; significant: 248


Letting the relationship vary over space improves the fit considerably: R² rises from 0.29 to 0.65.

- **Both terms are local.** The bandwidth of the intercept is 44 nearest municipalities (95% interval 44 to 48), the smallest the search considers, and that of the slope on lights is 65 (64 to 85), against 339 units. The level of poverty at given lights and its link with lights both change across the country.
- **One sign, many strengths.** All 339 local slopes are negative, from −1.35 to −0.17 standard deviations of poverty per standard deviation of lights (the global slope on that scale is the correlation, −0.54). After the correction for multiple testing, 260 are significant.
- The relationship is **steepest** in the central-southern Andean valleys, with department means of −1.23 in Cochabamba, −1.02 in Chuquisaca and −0.92 in Potosí, and every slope of Cochabamba and Chuquisaca significant.
- It is **flattest** in Beni (−0.29), Pando (−0.28) and La Paz (−0.41). Fewer than half of the slopes are significant in Beni (6 of 19) and Santa Cruz (21 of 56).
- The local condition numbers are small (at most 4.9, against a threshold of about 30), so the local coefficients are well identified.

The same model for 2024 keeps the intercept local (44) and finds a wider bandwidth for the slope (101, interval 101 to 112); it raises R² from 0.25 to 0.54, and again every local slope is negative (248 of them significant). A single global slope therefore understates the link in the valleys and overstates it in the Amazon and on the Altiplano. These are associations: a local slope can also reflect factors that vary over space and are not in the model. The `esda_heterogeneity` notebook reads this model in full, with the intercept, the fit against global models and the residuals.

## 7. Summary

- **Level, spread and change.** In 2012 the municipal MPI averages 0.23 and ranges from 0.018 to 0.56; by 2024 it has fallen in every unit, to an average of 0.06. Chuquisaca, Pando and Potosí are the poorest departments in 2012, Tarija and Santa Cruz the least poor, and the big cities the least poor municipalities. The ranking of municipalities hardly changes (rank correlation 0.86).
- **Correlates.** Poverty falls with electricity coverage and telephones (Spearman about −0.9), internet, night-time lights (−0.61), urbanization, financial access points and transit, and only weakly with population (−0.32). Lights alone account for 29% of its variation in 2012 and 25% in 2024.
- **Clustering.** Poverty is strongly clustered (Moran's I = 0.45 in 2012, 0.39 in 2024). Hot spots lie in the Chuquisaca–Potosí–Cochabamba valleys and, in 2012, in Pando; cold spots in Santa Cruz, the Cochabamba valley, the La Paz–El Alto area and Tarija. By 2024 the hot spots have retreated to the Andean core.
- **Neighbours' lights.** Poor municipalities tend to have dark neighbours (bivariate Moran's I = −0.27). Most hot spots with a significant bivariate pattern are poor places in dark neighbourhoods, and most cold spots less poor places in bright ones.
- **Inequality.** About four fifths of the inequality in poverty lies within departments, not between them, and relative inequality rose as poverty fell (Gini 0.25 in 2012, 0.37 in 2024).
- **Heterogeneity.** The lights–poverty slope is negative everywhere but its size varies about eightfold. It is steepest in the central-southern valleys and weakest in the Amazon, and it varies over short distances (a bandwidth of 65 of 339 municipalities in 2012).

All of these are associations between measured quantities, not causal effects. The methods are explained, with examples, in the package's [articles](https://quarcs-lab.github.io/geo4bolivia/articles/) and [function reference](https://quarcs-lab.github.io/geo4bolivia/reference/).